In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q datasets Pillow
!pip install -U datasets fsspec
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
   ━━━━━━━

In [ ]:
import json
import random
from datasets import load_dataset
from PIL import Image
import base64
import io
import gc
from google.colab import userdata
userdata.get('HF_TOKEN')

# 1. Load your Gemini teacher outputs
with open('hle_gemini-2.5-flash-lite.json', 'r') as f:
    teacher_data = json.load(f)

# 2. Load the HLE dataset (cais/hle is the official repo)
# HLE usually stores the ID in a 'row_id' or 'id' field; check the dataset preview
hle_ds = load_dataset("cais/hle", split="test")

def prepare_distillation_data(hle_dataset, teacher_outputs):
    formatted_data = []

    for example in hle_dataset:
        # Match the HLE row with your JSON key
        # HLE uses 'uuid' or 'id' - ensure this matches your JSON keys
        example_id = example.get('id') or example.get('uuid')

        if example_id in teacher_outputs:
            teacher_entry = teacher_outputs[example_id]

            if teacher_entry["usage"]["total_tokens"] <= 512:
              # PaliGemma SFT format: "cap <prompt>\n<answer>"
              # We strip the "Explanation:" prefix if you want just the logic + answer
              full_response = teacher_entry['response']

              formatted_data.append({
                  "id": example_id,
                  "image": example['image'], # PIL Image object from dataset
                  "prefix": f"cap {example['question']}\n",
                  "suffix": full_response
              })

    return formatted_data

# Process the data
full_set = prepare_distillation_data(hle_ds, teacher_data)

# 3. Perform 80/20 Split
random.seed(42) # For reproducibility
random.shuffle(full_set)

split_idx = int(len(full_set) * 0.8)
train_set = full_set[:split_idx]
test_set = full_set[split_idx:]

print(f"Total aligned examples: {len(full_set)}")
print(f"Training: {len(train_set)} | Testing: {len(test_set)}")

del hle_ds
del teacher_data
gc.collect() # Manually trigger Python garbage collection

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Total aligned examples: 388
Training: 310 | Testing: 78


30

In [ ]:
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "google/paligemma2-3b-pt-448"
adapter_path = "./distilled_adapter" # Path to where you saved the model

# 1. Load the same processor
processor = PaliGemmaProcessor.from_pretrained(model_id)

# 2. Load the base model in 4-bit
bnb_config_load = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config_load,
    device_map="auto"
)

# 3. Load the LoRA adapter onto the base model
loaded_model = PeftModel.from_pretrained(base_model, adapter_path)

preprocessor_config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

In [ ]:
from transformers import PaliGemmaProcessor
import numpy as np

model_id = "google/paligemma2-3b-pt-448"
processor = PaliGemmaProcessor.from_pretrained(model_id)

def decode_base64_to_image(base64_string):
    # 1. Remove the "data:image/jpeg;base64," prefix if it exists
    if "base64," in base64_string:
        base64_string = base64_string.split("base64,")[1]

    # 2. Decode the string into raw bytes
    img_bytes = base64.b64decode(base64_string)

    # 3. Use BytesIO to make the bytes "readable" like a file by PIL
    return Image.open(io.BytesIO(img_bytes)).convert("RGB")

def inference(example):
    img = example["image"]

    if isinstance(img, str):
        if img == "":
            img = Image.fromarray(
                np.zeros((224, 224, 3), dtype=np.uint8)
            ).convert("RGB")
        else:
            img = decode_base64_to_image(img)

    inputs = processor(
        text=example["prefix"],
        images=img,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = loaded_model.generate(**inputs, max_new_tokens=1536)

    decoded_output = processor.decode(output[0], skip_special_tokens=True)

    # --- token usage ---
    prompt_len = inputs["input_ids"].shape[-1]
    total_len = output.shape[-1]
    completion_len = total_len - prompt_len

    return {
        "response": decoded_output,
        "usage": {
            "prompt_tokens": int(prompt_len),
            "completion_tokens": int(completion_len),
            "total_tokens": int(total_len),
        }
    }

In [11]:
import json
from tqdm import tqdm

results = {}
model_name = "paligemma2-3b-pt-448"

for example in tqdm(test_set):
    example_id = example["id"]   # must match your JSON format

    output = inference(example)

    results[example_id] = {
        "model": model_name,
        "response": output["response"],
        "usage": output["usage"]
    }

100%|██████████| 78/78 [1:01:50<00:00, 47.57s/it]


In [12]:
output_path = "./finetuned_paligemma_results.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=4)